# Project Phase 2

## Short Introduction to our Dataset

### Original Research Question

*"How heavily did a state’s COVID-19 measurement policy have an impact on mental health
compared to its counterparts?"*

### Dataset Descriptions

- 1. CDC Depression Data *(data.cdc.gov)*: Surveys about the mental health of the U.S. population from 2020 until 2024. This was done by asking the experienced depression and anxiety of U.S. citizens and categorized by sex, ethnicity, state, and age.
- 2. Covid Policy Dataset *(github.com/OxCGRT)*: An Oxford study that broke down numerical representation of the U.S. country-wide and state specific Covid-related policies. Recorded from 2020 until 2022 and in multiple policy categories.

### Joins Explained

```python
merged = converted_depression_data.merge(
    oxcgrt_aggregated,
    left_on=['State', 'Time Period Start Date', 'Time Period End Date'],
    right_on=['RegionName', 'Start', 'End'],
    how='left'
)
```

We wanted to add the columnds from the Covid Policy Dataset to our CDC Depression Data. We did this for all unique combinations of specific U.S. states and the time period. Before we were able to do this, we needed to adjust the Covid Policy Dataset in multiple ways:

1) Fill 'United States' in the 'RegionName' column for all country-level data, as the Depression Dataset expects it in that format *(shown in step 1.3)*.
2) Group entries from daily entries to the time periods used in the Covid Policy Dataset *(shown in step 1.4)*.

After the merge we had all ~16,000 entries of the Depression dataset. 12,000 of them were extended with data from the Covid Policy Dataset, which covers all entries from 2020 until 2022. We decided to do a left join instead of an inner join, as it can give good insight about the depression and anxiety rates development after the pandemic, even when we don't have specific insight into the regulations at this time.

### Final Dataset Shape (rows x columns)

- Columns: 42
- Rows: 16,794


## Part 1 - Dataset Preparation and Joins

We used a left join on the depression dataset on the columns 'State', 'Time Period Start Date', 'Time Period End Date' with matching columns on the Covid policy dataset. We needed to create matching start and end date columns on the Covid policy dataset first, as its records were on a daily basis, while the depression dataset recorded their findings in specific time periods. We did a left join, because the depression dataset contains the main information components that we want to use, is more granular in categories (such as records by sex or ethnicity) and we just want to add some information about the government policies at that time from the other dataset.

The depression dataset originally had 14 columnns and grew by 28 columns. This happened as we added a majority of the columns of the Covid policy dataset. We filtered out some columns that we do not require or that already exist in the depression dataset, else we would have added even more columns.

### 1.1 Download the datasets

In [13]:
# Download the required datasets
import os
import urllib.request

os.makedirs("dataset", exist_ok=True)

file_path_1 = "dataset/cdc_depression_data.csv"
file_path_2 = "dataset/OxCGRT_simplified_v1.csv"
file_path_3 = "dataset/merged_dataset.csv"
url_path_1 = "https://data.cdc.gov/api/views/8pt5-q6wp/rows.csv?accessType=DOWNLOAD"
url_path_2 = "https://github.com/OxCGRT/covid-policy-dataset.git"

if not os.path.exists(file_path_1):
    print("Downloading CDC depression dataset from data.gov ...")
    urllib.request.urlretrieve(
        url_path_1,
        file_path_1
    )
else:
    print("Dataset already exists. Skipping download.")

if not os.path.exists(file_path_2):
    print("Downloading OxCGRT dataset from GitHub ...")
    # Clone the repository and move the file
    os.system("git clone " + url_path_2)
    os.system("mv covid-policy-dataset/data/OxCGRT_simplified_v1.csv dataset/")
    os.system("rm -rf covid-policy-dataset")
else:
    print("Dataset already exists. Skipping download.")


Cloning into 'covid-policy-dataset'...
Updating files: 100% (31/31), done.


### 1.2 Load the Datasets using Pandas

In [14]:
# Load the datasets
import pandas as pd
import numpy as np

depression_data = pd.read_csv(file_path_1)
oxcgrt_data = pd.read_csv(file_path_2)

/tmp/ipykernel_77122/3739561392.py:6: DtypeWarning: Columns (0: RegionName, 1: RegionCode, 2: MajorityVaccinated, 3: PopulationVaccinated) have mixed types. Specify dtype option on import or set low_memory=False.
  oxcgrt_data = pd.read_csv(file_path_2)


### 1.3 - Clean up and prepare our OxCGRT Dataset

In [15]:
# Filter the simplified OxCGRT dataset for the United States
oxcgrt_us = oxcgrt_data[oxcgrt_data['CountryName'] == 'United States']

# Remove some unnecessary columns that we won't be using for our analysis
columns_to_drop = ['CountryName', 'CountryCode', 'RegionCode', 'Jurisdiction']
oxcgrt_us = oxcgrt_us.drop(columns=columns_to_drop)

# Our dataset contains two versions of the measurements: *_combined and *_combined_numeric.
# For this assignment we will only keep the *_combined_numeric columns, as they are easier to work with for analysis and visualization.
# We might want to include the *_combined columns in a future assignment when we do more detailed analysis
columns_to_drop = [col for col in oxcgrt_us.columns if col.endswith('_combined') and not col.endswith('_combined_numeric')]
oxcgrt_us = oxcgrt_us.drop(columns=columns_to_drop)

# Fill missing values in the 'RegionName' column with 'United States'
# Required for merging with the depression dataset, which does expect 'United States' for country-level data
oxcgrt_us.loc[oxcgrt_us['RegionName'].isna() | (oxcgrt_us['RegionName'] == ''), 'RegionName'] = 'United States'


### 1.4 Aggregate the OxCGRT Dataset to match the Depression Datasets' Time Periods

In [16]:
# Get unique time periods and state combinations from the depression dataset
unique_time_state_combs = depression_data[['Time Period Start Date', 'Time Period End Date', 'State']].drop_duplicates()

# Convert 'Time Period Start Date' and 'Time Period End Date' to int YYYYMMDD format
unique_time_state_combs['Time Period Start Date'] = pd.to_datetime(
    unique_time_state_combs['Time Period Start Date'], format='%m/%d/%Y'
).dt.strftime('%Y%m%d').astype('int64')
unique_time_state_combs['Time Period End Date'] = pd.to_datetime(
    unique_time_state_combs['Time Period End Date'], format='%m/%d/%Y', errors='coerce'
).dt.strftime('%Y%m%d').astype('int64')

# Sort the rows of the OxCGRT dataset into groups, based on the unique time-state combinations from the depression dataset
# Add 'Start' and 'End' columns to the OxCGRT dataset and fill them based on the unique time-state combinations
# These will be matched to the depression dataset later when we do the merge, so we can easily filter OxCGRT data for the relevant time periods and states
oxcgrt_us['Start'] = np.nan
oxcgrt_us['End'] = np.nan

for start, end, state in unique_time_state_combs.values:
    mask = (
        (oxcgrt_us['RegionName'] == state) &
        (oxcgrt_us['Date'] >= start) &
        (oxcgrt_us['Date'] <= end)
    )
    oxcgrt_us.loc[mask, 'Start'] = start
    oxcgrt_us.loc[mask, 'End'] = end

# Now we can drop the 'Date' column, as we have the 'Start' and 'End' columns to indicate the relevant time periods for each row
oxcgrt_us = oxcgrt_us.drop(columns=['Date'])

# Convert the 'PopulationVaccinated' column to numeric
# (It was imported as an object, need to convert it to numeric for aggregation)
oxcgrt_us['PopulationVaccinated'] = pd.to_numeric(oxcgrt_us['PopulationVaccinated'])

# Aggregate the OxCGRT data by group (unique time-state combinations) using median for numeric columns
oxcgrt_aggregated = (
    oxcgrt_us
    .dropna(subset=['Start', 'End'])
    .groupby(['RegionName', 'Start', 'End'], as_index=False)
    .median(numeric_only=True)
)

# Add the 'MajorityVaccinated' column back to the aggregated dataset
oxcgrt_aggregated['MajorityVaccinated'] = (
    oxcgrt_us
    .groupby(['RegionName', 'Start', 'End'])['MajorityVaccinated']
    .first()
    .values
)


### 1.5 Merge the two Datasets

In [22]:
# Now we can merge the depression dataset with the aggregated OxCGRT dataset based on the unique time-state combinations
# Beforehand we need to convert the 'Time Period Start Date' and 'Time Period End Date' columns in the depression dataset (such as in the cell before)
converted_depression_data = depression_data.copy()
converted_depression_data['Time Period Start Date'] = pd.to_datetime(
    converted_depression_data['Time Period Start Date'], format='%m/%d/%Y'
).dt.strftime('%Y%m%d').astype('int64')
converted_depression_data['Time Period End Date'] = pd.to_datetime(
    converted_depression_data['Time Period End Date'], format='%m/%d/%Y', errors='coerce'
).dt.strftime('%Y%m%d').astype('int64')

rows_before_left = len(converted_depression_data)
rows_before_right = len(oxcgrt_aggregated)

# Merge the datasets based on the state and time period columns
merged = converted_depression_data.merge(
    oxcgrt_aggregated,
    left_on=['State', 'Time Period Start Date', 'Time Period End Date'],
    right_on=['RegionName', 'Start', 'End'],
    how='left'
)

rows_after = len(merged)

print("Rows before merge (depression):", rows_before_left)
print("Rows before merge (policy aggregated):", rows_before_right)
print("Rows after merge:", rows_after)

# Clean up merged dataset
merged = merged.drop(columns=['RegionName', 'Start', 'End']) # Redundant after merge
# merged = merged.drop(columns=['Time Period Start Date', 'Time Period End Date']) # Optional, as the Time Period Label already indicates the time period

# Save the merged dataset to a new CSV file
merged.to_csv(file_path_3, index=False)
print("Final merged dataset shape (rows, cols):", merged.shape)


Rows before merge (depression): 16794
Rows before merge (policy aggregated): 2661
Rows after merge: 16794
Final merged dataset shape (rows, cols): (16794, 42)


### 1.6 Describe the original Datasets and the merged Dataset for comparison

In [18]:
print("Depression dataset:")
print(depression_data.shape)
print(depression_data.info())
print(depression_data.describe())
print("##########################################################################################")

print("\nOxCGRT dataset (filtered for United States):")
print(oxcgrt_us.shape)
print(oxcgrt_us.info())
print(oxcgrt_us.describe())
print("##########################################################################################")

print("\nMerged dataset:")
print(merged.shape)
print(merged.info())
print(merged.describe())


Depression dataset:
(16794, 14)
<class 'pandas.DataFrame'>
RangeIndex: 16794 entries, 0 to 16793
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Indicator               16794 non-null  str    
 1   Group                   16794 non-null  str    
 2   State                   16794 non-null  str    
 3   Subgroup                16794 non-null  str    
 4   Phase                   16794 non-null  str    
 5   Time Period             16794 non-null  int64  
 6   Time Period Label       16794 non-null  str    
 7   Time Period Start Date  16794 non-null  str    
 8   Time Period End Date    16794 non-null  str    
 9   Value                   16087 non-null  float64
 10  Low CI                  16087 non-null  float64
 11  High CI                 16087 non-null  float64
 12  Confidence Interval     16087 non-null  str    
 13  Quartile Range          11017 non-null  str    
dtypes: float64(3), in

## Part 2: Exploratory Questions + Part 3: Required Interpretation

You must answer at least four specific EDA questions using visualization.

Each question must:

- Be clearly stated
- Use Pandas operations
- Include one iplot() visualization
- Include a written interpretation of 3 to 5 full sentences

You may not submit screenshots without explanation.

### Part 3: Required Interpretation

For each visualization, answer the following in full sentences:

1. What question are you asking?
2. What method did you use?
3. What does the visualization show?
4. What insight can you draw from it?

Minimum 3 sentences per visualization.  
Clarity and reasoning matter more than aesthetics.

---

**Optional - for future reports**

- Identifying rows lost during joins
- Using Boolean filtering or query()
- Identifying missing data patterns


In [ ]:
# **1. Category Counts (Bar Chart)**

# Example structure:

# - df[‘category’].value_counts().iplot(kind=‘bar’)

# Question type example:

# - Which category appears most frequently?

# [Category Counts] Vaccinated/Not Vaccinated for each year - Comparison

In [ ]:
# **2. Grouped Aggregation (Bar Chart)**

# Example structure:

# - df.groupby(‘group’)[‘numeric’].mean().iplot(kind=‘bar’)

# Question type example:

# - Which group has the highest average value?

# [Grouped Aggregation] Compare different age/ethnicities - average depression/anxiety as bar chart

In [ ]:
# **3. Distribution (Histogram)**

# Example structure:

# - df[‘numeric’].iplot(kind=‘hist’)

# Question type example:

# - Is the distribution symmetric or skewed?
# - Are there potential outliers?

In [ ]:
# **4. Trend or Top-N Comparison**

# If time or ordered data exists:

# - df.groupby(‘year’)[‘numeric’].mean().iplot(kind=‘line’)

# OR
# - df.sort_values(‘numeric’, ascending=False).head(10).iplot(kind=‘bar’)

# Question type example:

# - How does the main variable change over time?
# - What are the top 10 highest values?

# [Trend Comparison] 18-29 years and 40-49 year olds and compare them: Take the ConfirmedCases & ConfirmedDeaths to compare